**PORTADA**
---
1.  **Estudiante:** Patricio Gonzales Barboza
2.  **Modulo:** Analisis de datos con Python y SQL
3. **Proyecto:** Analisis de venta de una empresa retail
4. **Fuente:** https://www.kaggle.com/datasets/krakenteach13/ventas-tienda

**OBJETIVO**
---
Realizar un análisis exploratorio de los datos (EDA) para comprender la estructura del conjunto de datos, evaluar su calidad, identificar patrones de ventas y obtener información relevante mediante estadística descriptiva, visualizaciones interactivas y consultas SQL utilizando DuckDB.


**1. CARGA DE DATASET**

En esta sección se importa el archivo utilizando Pandas y se verifica que los datos hayan sido cargados correctamente.

In [8]:
import pandas as pd
import plotly.express as px
import numpy as np

In [13]:
datos_ventas = pd.read_csv("/Productos_vendidos_portienda.csv", encoding='cp1252', sep=';', engine='python')

**Nota:**

*Limpieza de Caracteres Diacríticos (Tildes, Ñ)*

A menudo, es útil estandarizar el texto eliminando tildes o la 'ñ' para facilitar búsquedas, comparaciones o el procesamiento en algunas herramientas. Utilizaremos `unicodedata` para lograr esto.

In [15]:
import unicodedata

def remover_diacriticos(texto):
    if isinstance(texto, str):
        # Reemplazar caracteres problematicos que son subproductos de la codificacion
        # Basado en los ejemplos de errores de decodificacion:
        # '’' (right single quotation mark) a menudo se ve en lugar de 'í' o 'ú'
        # '—' (em dash) a menudo se ve en lugar de 'ó' o 'á'
        # 'Ž' (latin capital letter Z with caron) a menudo se ve en lugar de 'é'
        # '–' (en dash) a menudo se ve en lugar de 'ñ'

        texto = texto.replace('’', 'i')
        texto = texto.replace('—', 'o')
        texto = texto.replace('Ž', 'e') # e.g., JosŽ -> Jose
        texto = texto.replace('–', 'n') # e.g., ni–os -> ninos

        # Normalizar y remover diacriticos con unicodedata
        # La categoria 'Mn' incluye marcas no espaciadoras (diacriticos)
        texto = ''.join(c for c in unicodedata.normalize('NFKD', texto) if unicodedata.category(c) != 'Mn')
        return texto
    return texto

# Aplicar la función a las columnas de tipo 'object' (strings)
# Se asume que `datos_ventas` ya está cargado correctamente.
for col in datos_ventas.select_dtypes(include='object').columns:
    datos_ventas[col] = datos_ventas[col].apply(remover_diacriticos)

# Mostrar las primeras filas para verificar los cambios
display(datos_ventas.head())

,fecha,tienda,categoria de producto,vendedor,producto,cantidad,precio,total
0,1/02/2026,Tienda B,Juguetes,Lucia,Puzzle 1000 piezas,6,625.01,3750.06
1,1/02/2026,Tienda E,Hogar,Maria,Cafetera Delonghi,1,84.42,84.42
2,1/03/2026,Tienda C,Ropa,Maria,Pantalon Levi's,5,560.97,2804.85
3,1/04/2026,Tienda D,Electronica,Ana,Smartphone Samsung,10,1197.81,11978.10
4,1/05/2026,Tienda E,Hogar,Maria,Ventilador Imaco,2,349.98,699.96


**2. EXPLORACIÓN INICIAL**



In [16]:
datos_ventas

,fecha,tienda,categoria de producto,vendedor,producto,cantidad,precio,total
0,1/02/2026,Tienda B,Juguetes,Lucia,Puzzle 1000 piezas,6,625.01,3750.06
1,1/02/2026,Tienda E,Hogar,Maria,Cafetera Delonghi,1,84.42,84.42
2,1/03/2026,Tienda C,Ropa,Maria,Pantalon Levi's,5,560.97,2804.85
3,1/04/2026,Tienda D,Electronica,Ana,Smartphone Samsung,10,1197.81,11978.10
4,1/05/2026,Tienda E,Hogar,Maria,Ventilador Imaco,2,349.98,699.96
...,...,...,...,...,...,...,...,...
495,5/11/2027,Tienda B,Electronica,Lucia,Smartwatch Apple,9,1154.35,10389.15
496,5/12/2027,Tienda E,Juguetes,Jose,Drone para ninos,10,482.63,4826.30
497,5/13/2027,Tienda C,Ropa,Maria,Zapatillas Nike,4,541.64,2166.56
498,5/14/2027,Tienda D,Juguetes,Rosa,Lego Star Wars,3,1388.01,4164.03


In [17]:
datos_ventas.shape

(500, 8)

In [18]:
datos_ventas.head()

,fecha,tienda,categoria de producto,vendedor,producto,cantidad,precio,total
0,1/02/2026,Tienda B,Juguetes,Lucia,Puzzle 1000 piezas,6,625.01,3750.06
1,1/02/2026,Tienda E,Hogar,Maria,Cafetera Delonghi,1,84.42,84.42
2,1/03/2026,Tienda C,Ropa,Maria,Pantalon Levi's,5,560.97,2804.85
3,1/04/2026,Tienda D,Electronica,Ana,Smartphone Samsung,10,1197.81,11978.10
4,1/05/2026,Tienda E,Hogar,Maria,Ventilador Imaco,2,349.98,699.96


In [ ]:
datos_ventas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   fecha                  500 non-null    object 
 1   tienda                 500 non-null    object 
 2   categoria de producto  500 non-null    object 
 3   vendedor               500 non-null    object 
 4   producto               500 non-null    object 
 5   cantidad               500 non-null    int64  
 6   precio                 500 non-null    float64
 7   total                  500 non-null    float64
dtypes: float64(2), int64(1), object(5)
memory usage: 31.4+ KB


In [ ]:
datos_ventas.tail()

,fecha,tienda,categoria de producto,vendedor,producto,cantidad,precio,total
495,5/11/2027,Tienda B,Electronica,Lucia,Smartwatch Apple,9,1154.35,10389.15
496,5/12/2027,Tienda E,Juguetes,Jose,Drone para ninos,10,482.63,4826.30
497,5/13/2027,Tienda C,Ropa,Maria,Zapatillas Nike,4,541.64,2166.56
498,5/14/2027,Tienda D,Juguetes,Rosa,Lego Star Wars,3,1388.01,4164.03
499,5/15/2027,Tienda D,Ropa,Carlos,Casaca Puma,4,318.24,1272.96


**3. ANÁLISIS DE CALIDAD DE DATOS**



In [21]:
datos_ventas.isnull()

,fecha,tienda,categoria de producto,vendedor,producto,cantidad,precio,total
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
495,False,False,False,False,False,False,False,False
496,False,False,False,False,False,False,False,False
497,False,False,False,False,False,False,False,False
498,False,False,False,False,False,False,False,False


In [22]:
datos_ventas.isnull().sum()

,0
fecha,0
tienda,0
categoria de producto,0
vendedor,0
producto,0
cantidad,0
precio,0
total,0


In [24]:
datos_ventas['fecha'] = pd.to_datetime(datos_ventas['fecha'], format='%m/%d/%Y')
display(datos_ventas.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   fecha                  500 non-null    datetime64[ns]
 1   tienda                 500 non-null    object        
 2   categoria de producto  500 non-null    object        
 3   vendedor               500 non-null    object        
 4   producto               500 non-null    object        
 5   cantidad               500 non-null    int64         
 6   precio                 500 non-null    float64       
 7   total                  500 non-null    float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 31.4+ KB


None

In [23]:
datos_ventas.duplicated().sum()

np.int64(0)

**4. ESTADÍSTICA DESCRIPTIVA**

In [ ]:
datos_ventas.describe()

,cantidad,precio,total
count,500.000000,500.000000,500.000000
mean,5.388000,751.207640,4070.289460
std,2.863189,442.264961,3443.878454
min,1.000000,20.370000,26.690000
25%,3.000000,352.822500,1248.155000
50%,5.000000,753.405000,3016.300000
75%,8.000000,1144.400000,6100.590000
max,10.000000,1499.620000,14946.300000


In [ ]:
datos_ventas.describe(include="object")

,fecha,tienda,categoria de producto,vendedor,producto
count,500,500,500,500,500
unique,499,5,6,8,30
top,1/02/2026,Tienda D,Ropa,Lucia,Casaca Puma
freq,2,108,92,72,23


In [ ]:
datos_ventas["categoria de producto"].value_counts()

,count
categoria de producto,
Ropa,92
Hogar,90
Electronica,84
Juguetes,82
Belleza,79
Deportes,73


In [ ]:
datos_ventas.groupby("tienda")["total"].sum().apply(lambda x: f"${x:,.2f}")

,total
tienda,
Tienda A,"$424,228.05"
Tienda B,"$384,941.41"
Tienda C,"$331,483.41"
Tienda D,"$431,107.13"
Tienda E,"$463,384.73"


**5. VISUALIZACIONES EN PLOTLY**

In [20]:
grafico_ventas = datos_ventas.groupby("tienda")["total"].sum().reset_index()

px.bar(
    datos_ventas,
    x="tienda",
    y="total",
    title="Ventas por tienda"
)

In [ ]:
ventas_categoria = datos_ventas.groupby("categoria de producto")["total"].sum().reset_index()

fig = px.bar(
    ventas_categoria,
    x="categoria de producto",
    y="total",
    title="Ventas por Categoria de Producto"
)
fig.show()

In [93]:
ventas_vendedor = datos_ventas.groupby("vendedor")["total"].sum().reset_index()

fig = px.bar(
    ventas_vendedor,
    x="vendedor",
    y="total",
    title="Ventas por Vendedor"
)
fig.show()

In [98]:
cantidad_categoria = datos_ventas.groupby("categoria de producto")["cantidad"].sum().reset_index()

fig = px.bar(
    cantidad_categoria,
    x="categoria de producto",
    y="cantidad",
    title="Cantidades Vendidas por Categoria"
)
fig.show()

Ahora la columna 'fecha' ha sido convertida al tipo de dato `datetime64[ns]`, lo que permitirá realizar análisis de tiempo correctamente.

In [25]:
# Crear una columna que combine el año y el mes para la agrupación
datos_ventas['mes_anio'] = datos_ventas['fecha'].dt.to_period('M')

# Agrupar las ventas por mes/año y sumar los totales
ventas_mensuales = datos_ventas.groupby('mes_anio')['total'].sum().reset_index()

# Convertir 'mes_anio' a tipo string para Plotly
ventas_mensuales['mes_anio'] = ventas_mensuales['mes_anio'].astype(str)

# Crear el gráfico de tendencia
fig = px.line(
    ventas_mensuales,
    x="mes_anio",
    y="total",
    title="Tendencia de Ventas Mensual/Anual",
    markers=True
)

fig.update_layout(xaxis_title="Mes/Año", yaxis_title="Total de Ventas")
fig.show()